In [12]:
import serial
import time
import datetime as dt

In [13]:
DEVICE = 'COM8'
BAUDRATE = 115200
TIMEOUT = 1
ADDRESS = '0'
# ====================================

STX = '\x02'
ETX = '\x03'
SEP = ':'

In [14]:
def checksum(payload: str) -> str:
    x = 0
    for ch in payload:
        x ^= ord(ch)
    return format(x, '02X')


def build_command(address: str, cmd: str) -> bytes:
    body = f"{address}{cmd}{SEP}"
    chksum = checksum(body)
    return (STX + body + chksum + ETX).encode('latin1')


def read_response(port: serial.Serial) -> bytes:
    port.timeout = TIMEOUT
    raw = port.read_until(ETX.encode("latin1"))
    if not raw:
        return b""
    return raw

def send_cmd(port: serial.Serial, cmd: str) -> bytes:
    frame = build_command(ADDRESS, cmd)
    port.write(frame)
    return read_response(port)

def parse_txpulse(rx: bytes | str) -> int | None:
    if not rx:
        return None

    text = rx.decode("latin1") if isinstance(rx, bytes) else rx
    pulse_text = text[2:-1].split(":")[0]

    if pulse_text == "":
        return None

    return int(pulse_text)

def send_cmd(port: serial.Serial, cmd: str) -> bytes:
    frame = build_command(ADDRESS, cmd)
    port.write(frame)
    return read_response(port)

def send_cmdwr(port, cmd, timeout_threshold=TIMEOUT):
    """
    Sends a command and retries immediately if:
    1. The response is empty (Serial Timeout).
    2. The response time (dt) exceeds the threshold.
    """
    while True:
        t0 = time.perf_counter()
        rx = send_cmd(port, cmd)
        dt = time.perf_counter() - t0

        if not rx or dt >= timeout_threshold:
            # Logic for retry
            print(f"[RETRY]    {cmd:20s}  {dt:.3f} s  rx={rx!r}")
            # Optional: time.sleep(0.05) to give the bus a breather
            continue 
        else:
            # Logic for success
            print(f"[OK]       {cmd:20s}  {dt:.3f} s  rx={rx!r}")
            return rx

In [15]:
'''# --- Constants ---
MODULES = [1, 2, 3]
AXIS = 1

CURRENT = 20
FREQUENCY = 200



# --- Main Logic ---
port = serial.Serial(DEVICE, BAUDRATE, timeout=TIMEOUT)

try:
    if port.is_open:
        for MODULE in MODULES:
            # Initialization
            send_cmdwr(port, f"{MODULE}.{AXIS}C")
            send_cmdwr(port, f"{MODULE}.{AXIS}MD")
            send_cmdwr(port, f"M{MODULE}.{AXIS}P17=0")

            # Current Settings
            send_cmdwr(port, f"M{MODULE}.{AXIS}P40=0")
            send_cmdwr(port, f"M{MODULE}.{AXIS}P42=0")
            send_cmdwr(port, f"M{MODULE}.{AXIS}P41={CURRENT}")

            # Pulse / Frequency Settings
            send_cmdwr(port, f"M{MODULE}.{AXIS}P45=0")
            send_cmdwr(port, f"M{MODULE}.{AXIS}P14={FREQUENCY}")
            
            # Counter Reset & Activation
            send_cmdwr(port, f"M{MODULE}.{AXIS}P20S0")
            send_cmdwr(port, "SAE")
            send_cmdwr(port, f"{MODULE}.{AXIS}MA")

            # Movement
            send_cmdwr(port, f"M{MODULE}.{AXIS}-3400")

finally:
    if port.is_open:
        port.close()
        print("Port closed.")'''

'# --- Constants ---\nMODULES = [1, 2, 3]\nAXIS = 1\n\nCURRENT = 20\nFREQUENCY = 200\n\n\n\n# --- Main Logic ---\nport = serial.Serial(DEVICE, BAUDRATE, timeout=TIMEOUT)\n\ntry:\n    if port.is_open:\n        for MODULE in MODULES:\n            # Initialization\n            send_cmdwr(port, f"{MODULE}.{AXIS}C")\n            send_cmdwr(port, f"{MODULE}.{AXIS}MD")\n            send_cmdwr(port, f"M{MODULE}.{AXIS}P17=0")\n\n            # Current Settings\n            send_cmdwr(port, f"M{MODULE}.{AXIS}P40=0")\n            send_cmdwr(port, f"M{MODULE}.{AXIS}P42=0")\n            send_cmdwr(port, f"M{MODULE}.{AXIS}P41={CURRENT}")\n\n            # Pulse / Frequency Settings\n            send_cmdwr(port, f"M{MODULE}.{AXIS}P45=0")\n            send_cmdwr(port, f"M{MODULE}.{AXIS}P14={FREQUENCY}")\n            \n            # Counter Reset & Activation\n            send_cmdwr(port, f"M{MODULE}.{AXIS}P20S0")\n            send_cmdwr(port, "SAE")\n            send_cmdwr(port, f"{MODULE}.{AXIS}MA")\n\

In [16]:
# --- Constants ---
DEVICE = 'COM8'
BAUDRATE = 115200
TIMEOUT = 1  # Standard timeout for the port
MODULES = [1, 2, 3]
AXIS = 1
CURRENT = 100
FREQUENCY = 200

# --- Main Logic ---
port = serial.Serial(DEVICE, BAUDRATE, timeout=TIMEOUT, parity="N", stopbits=1)

if port.is_open:
    print("Successfully opened")

try:
    for MODULE in MODULES:
        # Reset and deactivate
        send_cmdwr(port, f"{MODULE}.{AXIS}C")
        send_cmdwr(port, f"{MODULE}.{AXIS}MD")

        # Function settings
        send_cmdwr(port, f"{MODULE}.{AXIS}P17={0}")  # Boost off

        # Current settings
        send_cmdwr(port, f"{MODULE}.{AXIS}MD")       # Deactivate again to be safe
        send_cmdwr(port, f"{MODULE}.{AXIS}P40={20}")  # Stop current off
        send_cmdwr(port, f"{MODULE}.{AXIS}P41={CURRENT}")
        send_cmdwr(port, f"{MODULE}.{AXIS}P42={0}")  # Boost current off
        send_cmdwr(port, f"{MODULE}.{AXIS}P45={0}")  # Boost current off

        # Pulse settings
        send_cmdwr(port, f"{MODULE}.{AXIS}P14={FREQUENCY}")

        # Reset pulse counter
        send_cmdwr(port, f"{MODULE}.{AXIS}P20S0")

        # Sequence and Activation
        rx = send_cmdwr(port, f"{MODULE}.{AXIS}P20R")
        pulse = parse_txpulse(rx) # Assuming this function is defined elsewhere
        
        send_cmdwr(port, "SAE")                     # Set params
        send_cmdwr(port, f"{MODULE}.{AXIS}MA")       # Activate

finally:
    port.close()

# Verification of closure
time.sleep(5)
if not port.is_open:
    print(f"Successfully closed {DEVICE}")

Successfully opened
[OK]       1.1C                  0.021 s  rx=b'\x02\x06:3C\x03'
[OK]       1.1MD                 0.015 s  rx=b'\x02\x06:3C\x03'
[OK]       1.1P17=0              0.016 s  rx=b'\x02\x06:3C\x03'
[OK]       1.1MD                 0.016 s  rx=b'\x02\x06:3C\x03'
[OK]       1.1P40=20             0.016 s  rx=b'\x02\x06:3C\x03'
[RETRY]    1.1P41=100            1.005 s  rx=b''
[OK]       1.1P41=100            0.017 s  rx=b'\x02\x06:3C\x03'
[OK]       1.1P42=0              0.016 s  rx=b'\x02\x06:3C\x03'
[OK]       1.1P45=0              0.016 s  rx=b'\x02\x06:3C\x03'
[RETRY]    1.1P14=200            1.004 s  rx=b''
[OK]       1.1P14=200            0.019 s  rx=b'\x02\x06:3C\x03'
[OK]       1.1P20S0              0.016 s  rx=b'\x02\x06:3C\x03'
[RETRY]    1.1P20R               1.003 s  rx=b''
[OK]       1.1P20R               0.005 s  rx=b'\x02\x060:0C\x03'
[OK]       SAE                   0.014 s  rx=b'\x02\x15:2F\x03'
[OK]       1.1MA                 0.016 s  rx=b'\x02\x06:3C\x03'


ここまで！！！！

In [17]:
port = serial.Serial(DEVICE, BAUDRATE, timeout=TIMEOUT, parity="N", stopbits=1)

if port.is_open:
    print("Successfully opened")

try:
    # --- Start Movement Sequence ---
    _ = send_cmdwr(port, "S1")
    _ = send_cmdwr(port, "1.1-1600 2.1-1600 3.1-1600")
    #_ = send_cmdwr(port, "1.1+2300 2.1+2300 3.1+2300")
    #_ = send_cmdwr(port, "1.1-2300 2.1-2300 3.1-2300")
        #_ = send_cmdwr(port, "1.1-93 2.1-93 3.1-93")
        #_ = send_cmdwr(port, "2.1+200 3.1+200")
    _ = send_cmdwr(port, "S0")

    
finally:
    port.close()

time.sleep(1)
if not port.is_open:
    print(f"Successfully closed {DEVICE}")

Successfully opened
[OK]       S1                    0.006 s  rx=b'\x02\x06:3C\x03'
[OK]       1.1-1600 2.1-1600 3.1-1600  0.013 s  rx=b'\x02\x06\x06\x06:3C\x03'
[OK]       S0                    0.016 s  rx=b'\x02\x06:3C\x03'
Successfully closed COM8
